In [1]:
import pandas as pd
import numpy as np
import logging
from typing import Tuple, Optional

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def calculate_quant_metrics(price_series: pd.Series, window: int = 252) -> Optional[pd.DataFrame]:
    """
    단일 자산의 일일 종가를 기반으로 연간 변동성(Volatility) 및 MDD를 계산
    """
    try:
        if price_series.empty:
            raise ValueError("입력된 price_series 데이터가 비어 있습니다.")

        df = price_series.to_frame(name='close')
        
        # Log Returns 계산
        df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
        
        # Annualized Volatility
        df['volatility'] = df['log_ret'].rolling(window=window).std() * np.sqrt(252)
        
        # MDD (Maximum Drawdown) 계산
        cum_ret = (1 + df['log_ret'].dropna()).cumprod()
        rolling_max = cum_ret.cummax()
        df['drawdown'] = cum_ret / rolling_max - 1.0
        df['mdd'] = df['drawdown'].rolling(window=window).min()
        
        logging.info(f"메트릭 계산 완료. (총 {len(df)} 거래일)")
        return df

    except Exception as e:
        logging.error(f"메트릭 계산 중 오류 발생: {e}")
        return None

def calculate_spread_zscore(asset_y: pd.Series, asset_x: pd.Series, window: int = 20) -> Optional[pd.Series]:
    """
    Pairs Trading(통계적 차익거래)을 위한 두 자산 간 스프레드의 Z-Score 계산
    """
    try:
        # 단순 비율(Ratio) 기반 스프레드 (실제 퀀트에서는 OLS 선형회귀 잔차를 사용)
        spread = np.log(asset_y / asset_x)
        
        spread_mean = spread.rolling(window=window).mean()
        spread_std = spread.rolling(window=window).std()
        
        z_score = (spread - spread_mean) / spread_std
        
        logging.info(f"Z-Score 계산 완료 (window={window})")
        return z_score

    except Exception as e:
        logging.error(f"Z-Score 계산 중 오류 발생: {e}")
        return None

# 실행 예시
if __name__ == "__main__":
    # Mock 데이터 생성 (Random Walk)
    np.random.seed(42)
    dates = pd.date_range(start='2020-01-01', periods=1000, freq='B')
    
    # Asset A & B (Cointegrated 가정)
    asset_a = pd.Series(np.exp(np.random.normal(0, 0.01, 1000).cumsum()) * 100, index=dates)
    asset_b = asset_a * 1.5 + np.random.normal(0, 2, 1000)

    # 1. 단일 자산 리스크 매트릭스 도출
    metrics_df = calculate_quant_metrics(asset_a)
    if metrics_df is not None:
        print(f"최대 낙폭(MDD): {metrics_df['mdd'].min():.2%}")

    # 2. Pairs Z-Score 도출 (Z-Score > 2 or < -2 일 때 진입 시그널)
    z_scores = calculate_spread_zscore(asset_a, asset_b)
    if z_scores is not None:
        print(f"최근 Z-Score: {z_scores.iloc[-1]:.4f}")
        

2026-05-26 09:58:13,755 - INFO - 메트릭 계산 완료. (총 1000 거래일)
2026-05-26 09:58:13,759 - INFO - Z-Score 계산 완료 (window=20)


최대 낙폭(MDD): -28.80%
최근 Z-Score: 1.1750


In [2]:
import yfinance as yf
import pandas as pd
import logging
from typing import List, Optional
from datetime import datetime

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def scan_sector_momentum(tickers: List[str]) -> Optional[pd.DataFrame]:
    """
    지정된 ETF 티커 목록의 가격 데이터를 수집하고 MA 크로스오버 및 RSI 지표를 계산합니다.
    """
    results = []
    
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            # 최근 6개월 데이터 추출
            hist = stock.history(period="6mo")
            
            if hist.empty:
                logging.warning(f"[{ticker}] 데이터를 불러올 수 없습니다.")
                continue

            # 이동평균선 (20일, 50일)
            hist['MA_20'] = hist['Close'].rolling(window=20).mean()
            hist['MA_50'] = hist['Close'].rolling(window=50).mean()

            # RSI (14일)
            delta = hist['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            hist['RSI'] = 100 - (100 / (1 + rs))

            latest = hist.iloc[-1]
            
            # 골든크로스(Bullish) / 데드크로스(Bearish) 판별
            trend = "Bullish" if latest['MA_20'] > latest['MA_50'] else "Bearish"

            results.append({
                'Ticker': ticker,
                'Close_Price': round(latest['Close'], 2),
                'Trend_20_50': trend,
                'RSI_14': round(latest['RSI'], 2),
                'Volume': int(latest['Volume'])
            })
            
        except Exception as e:
            logging.error(f"[{ticker}] 처리 중 예외 발생: {e}")
            continue

    if not results:
        return None
        
    df = pd.DataFrame(results)
    logging.info("섹터 모멘텀 스캔 완료.")
    return df

# 실행 예시
if __name__ == "__main__":
    # 타겟 섹터 ETF: 로보틱스/자동화(BOTZ), 사이버보안(CIBR), AI/테크(AIQ), 인프라(PAVE)
    target_sectors = ["BOTZ", "CIBR", "AIQ", "PAVE"]
    
    momentum_df = scan_sector_momentum(target_sectors)
    
    if momentum_df is not None:
        print(f"\n--- 섹터 모멘텀 리포트 ({datetime.now().strftime('%Y-%m-%d')}) ---")
        print(momentum_df.to_string(index=False))

2026-05-26 10:05:03,054 - INFO - 섹터 모멘텀 스캔 완료.



--- 섹터 모멘텀 리포트 (2026-05-26) ---
Ticker  Close_Price Trend_20_50  RSI_14  Volume
  BOTZ        40.29     Bullish   60.67  976500
  CIBR        84.28     Bullish   90.03 1597500
   AIQ        62.81     Bullish   70.30 1987400
  PAVE        54.94     Bullish   43.15 1224400


In [3]:
import yfinance as yf
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(message)s')

def calculate_entry_targets(tickers: list) -> None:
    """
    타겟 종목의 최적 지정가(Limit Order Price) 산출
    - 1차 매수: 20일 MA (단기 눌림목)
    - 2차 매수: 50일 MA (중기 추세 지지선)
    """
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="3mo")
            
            if hist.empty:
                continue

            current_price = hist['Close'].iloc[-1]
            ma_20 = hist['Close'].rolling(window=20).mean().iloc[-1]
            ma_50 = hist['Close'].rolling(window=50).mean().iloc[-1]
            
            # 현재가가 20일선 위에 있으면 단기 강세장으로 판단
            trend = "초강세 (추격 매수 주의)" if current_price > ma_20 else "조정 국면 (진입 기회)"
            
            print(f"\n[{ticker}] 현재가: ${current_price:.2f} ({trend})")
            print(f" ┣ 1차 매수 타점 (MA20): ${ma_20:.2f} (비중 30%)")
            print(f" ┗ 2차 매수 타점 (MA50): ${ma_50:.2f} (비중 70%)")

        except Exception as e:
            logging.error(f"{ticker} 데이터 처리 오류: {e}")

if __name__ == "__main__":
    target_stocks = ["VRT", "ANET", "SYM"]
    calculate_entry_targets(target_stocks)


[VRT] 현재가: $327.46 (조정 국면 (진입 기회))
 ┣ 1차 매수 타점 (MA20): $339.12 (비중 30%)
 ┗ 2차 매수 타점 (MA50): $303.02 (비중 70%)

[ANET] 현재가: $154.03 (초강세 (추격 매수 주의))
 ┣ 1차 매수 타점 (MA20): $153.06 (비중 30%)
 ┗ 2차 매수 타점 (MA50): $147.07 (비중 70%)

[SYM] 현재가: $54.03 (초강세 (추격 매수 주의))
 ┣ 1차 매수 타점 (MA20): $53.79 (비중 30%)
 ┗ 2차 매수 타점 (MA50): $54.46 (비중 70%)


In [4]:
import yfinance as yf
import pandas as pd
import logging

# 로깅 설정 (print 대신 logging 활용)
logging.basicConfig(level=logging.INFO, format='%(message)s')

def analyze_korean_stock(ticker: str, company_name: str) -> None:
    """
    한국 증시 단일 종목의 RSI 및 이동평균선 기반 매수 타점 분석
    """
    # 야후 파이낸스 한국 주식 포맷 (.KS = 코스피, .KQ = 코스닥)
    symbol = f"{ticker}.KS"
    
    try:
        stock = yf.Ticker(symbol)
        hist = stock.history(period="6mo")
        
        if hist.empty:
            logging.error(f"[{company_name}] 데이터 로드 실패. 티커를 확인하세요.")
            return

        # 기술적 지표 계산
        current_price = hist['Close'].iloc[-1]
        ma_20 = hist['Close'].rolling(window=20).mean().iloc[-1]
        ma_50 = hist['Close'].rolling(window=50).mean().iloc[-1]
        
        # RSI (14일) 산출
        delta = hist['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs)).iloc[-1]
        
        # 이격도 (Disparity)
        disparity_20 = (current_price / ma_20) * 100
        
        # 진입 판단 로직
        if rsi >= 70:
            status = "과매수 (신규 진입 보류, 조정 대기)"
        elif rsi <= 30:
            status = "과매도 (강력 매수 구간)"
        elif current_price > ma_20 and disparity_20 > 105:
            status = "단기 고평가 (MA20 이탈 대기)"
        else:
            status = "조정 국면 (분할 매수 유효)"
            
        logging.info(f"\n--- {company_name} ({ticker}) 퀀트 지표 리포트 ---")
        logging.info(f"현재가: {current_price:,.0f} KRW")
        logging.info(f"RSI (14일): {rsi:.1f} / 현재 상태: {status}")
        logging.info(f"20일선 이격도: {disparity_20:.1f}%\n")
        
        logging.info(f"[시스템 트레이딩 기반 추천 매수 타점]")
        logging.info(f" ┣ 1차 매수 (MA20): {ma_20:,.0f} KRW (비중 30% - 단기 눌림목)")
        logging.info(f" ┗ 2차 매수 (MA50): {ma_50:,.0f} KRW (비중 70% - 중장기 지지선)")

    except Exception as e:
        logging.error(f"데이터 분석 중 오류 발생: {e}")

if __name__ == "__main__":
    # 두산에너빌리티(034020) 단독 분석
    analyze_korean_stock("034020", "두산에너빌리티")

2026-05-26 10:12:59,711 - INFO - 
--- 두산에너빌리티 (034020) 퀀트 지표 리포트 ---
2026-05-26 10:12:59,712 - INFO - 현재가: 112,600 KRW
2026-05-26 10:12:59,713 - INFO - RSI (14일): 37.5 / 현재 상태: 조정 국면 (분할 매수 유효)
2026-05-26 10:12:59,713 - INFO - 20일선 이격도: 93.3%

2026-05-26 10:12:59,714 - INFO - [시스템 트레이딩 기반 추천 매수 타점]
2026-05-26 10:12:59,714 - INFO -  ┣ 1차 매수 (MA20): 120,690 KRW (비중 30% - 단기 눌림목)
2026-05-26 10:12:59,715 - INFO -  ┗ 2차 매수 (MA50): 110,286 KRW (비중 70% - 중장기 지지선)


In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import logging
from typing import Dict, Optional

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(message)s')

def calculate_capital_efficiency(tickers: Dict[str, str]) -> Optional[pd.DataFrame]:
    """
    국내 타겟 종목들의 자본 배치 효율성(Sharpe Ratio) 및 현재 진입 매력도(RSI)를 정량화
    """
    results = []
    
    for name, ticker in tickers.items():
        try:
            stock = yf.Ticker(ticker)
            # 최근 1년(252 거래일) 데이터 
            hist = stock.history(period="1y")
            
            if hist.empty:
                logging.warning(f"[{name}] 데이터 로드 실패. 티커를 확인하세요.")
                continue

            # 1. Sharpe Ratio (위험 조정 수익률) 계산
            # 일일 로그 수익률 계산
            hist['log_ret'] = np.log(hist['Close'] / hist['Close'].shift(1))
            # 연율화된 수익률 및 변동성
            annual_ret = hist['log_ret'].mean() * 252
            annual_vol = hist['log_ret'].std() * np.sqrt(252)
            
            # 무위험 수익률(Risk-Free Rate) 3.5% 가정
            risk_free_rate = 0.035
            sharpe_ratio = (annual_ret - risk_free_rate) / annual_vol

            # 2. 단기 진입 타점 지표 (RSI 및 이격도)
            current_price = hist['Close'].iloc[-1]
            ma_20 = hist['Close'].rolling(window=20).mean().iloc[-1]
            disparity_20 = (current_price / ma_20) * 100
            
            # RSI (14일)
            delta = hist['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            rsi_14 = 100 - (100 / (1 + rs)).iloc[-1]

            results.append({
                '종목명': name,
                'Sharpe_Ratio': round(sharpe_ratio, 2),
                '연간 변동성(%)': round(annual_vol * 100, 1),
                'RSI(14)': round(rsi_14, 1),
                '20일 이격도(%)': round(disparity_20, 1)
            })

        except Exception as e:
            logging.error(f"[{name}] 데이터 처리 중 에러 발생: {e}")

    if not results:
        return None

    df = pd.DataFrame(results)
    # Sharpe Ratio가 높은 순서대로 정렬 (과거 1년 가장 효율적인 자산)
    df = df.sort_values(by='Sharpe_Ratio', ascending=False)
    return df

if __name__ == "__main__":
    target_stocks = {
        "HD현대일렉트릭": "267260.KS",
        "이수페타시스": "036530.KS",
        "두산로보틱스": "454910.KS",
        "두산에너빌리티": "034020.KS"
    }
    
    efficiency_df = calculate_capital_efficiency(target_stocks)
    
    if efficiency_df is not None:
        logging.info("\n--- 자본 배치 효율성 리포트 (Sharpe Ratio 기준) ---")
        logging.info(efficiency_df.to_string(index=False))

2026-05-26 10:15:31,700 - INFO - 
--- 자본 배치 효율성 리포트 (Sharpe Ratio 기준) ---
2026-05-26 10:15:31,706 - INFO -      종목명  Sharpe_Ratio  연간 변동성(%)  RSI(14)  20일 이격도(%)
HD현대일렉트릭          1.93       57.7     38.5        93.0
 두산에너빌리티          1.51       66.5     37.2        93.1
  두산로보틱스          1.20       67.4     49.6       102.2
  이수페타시스          0.46       52.5     28.1        95.1


In [6]:
import yfinance as yf
import pandas as pd
import logging
from typing import Dict, Optional

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def fetch_deep_value_metrics(tickers: Dict[str, str]) -> Optional[pd.DataFrame]:
    """
    주어진 티커 목록의 핵심 퀀트 가치평가 지표(PER, PBR, ROE, Dividend Yield)를 추출합니다.
    """
    results = []
    
    for name, ticker in tickers.items():
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            
            # API에서 데이터를 가져오지 못할 경우 방어 로직 (기본값 0.0)
            trailing_pe = info.get('trailingPE', 0.0)
            price_to_book = info.get('priceToBook', 0.0)
            roe = info.get('returnOnEquity', 0.0) * 100 if info.get('returnOnEquity') else 0.0
            div_yield = info.get('dividendYield', 0.0) * 100 if info.get('dividendYield') else 0.0
            
            # 딥밸류 필터링 조건 평가
            is_deep_value = "Yes" if (0 < trailing_pe < 8) and (0 < price_to_book < 1.0) else "No"
            
            results.append({
                '종목명': name,
                'PER (배)': round(trailing_pe, 2),
                'PBR (배)': round(price_to_book, 2),
                'ROE (%)': round(roe, 2),
                '배당수익률 (%)': round(div_yield, 2),
                '극저평가 여부': is_deep_value
            })
            
        except Exception as e:
            logging.error(f"[{name}] 데이터 추출 중 예외 발생: {e}")
            continue

    if not results:
        return None
        
    df = pd.DataFrame(results)
    logging.info("가치평가 지표 추출 완료.")
    return df

if __name__ == "__main__":
    # 한국 증시 핵심 딥밸류 및 비교군 티커 설정
    target_stocks = {
        "현대차": "053800.KS",
        "KB금융": "105560.KS",
        "삼성전자": "005930.KS" # 비교를 위한 일반 우량주 편입
    }
    
    metrics_df = fetch_deep_value_metrics(target_stocks)
    
    if metrics_df is not None:
        print("\n--- Deep Value 퀀트 지표 리포트 ---")
        print(metrics_df.to_string(index=False))

2026-05-26 10:20:17,103 - INFO - 가치평가 지표 추출 완료.



--- Deep Value 퀀트 지표 리포트 ---
 종목명  PER (배)  PBR (배)  ROE (%)  배당수익률 (%) 극저평가 여부
 현대차      0.0      0.0     0.00        0.0      No
KB금융      0.0      0.0     9.99      287.0      No
삼성전자      0.0      0.0    18.86       51.0      No


In [9]:
# 필요 라이브러리 설치: pip install pykrx
from pykrx import stock
from datetime import datetime
import logging
import pandas as pd
from typing import Dict, Optional

def fetch_krx_fundamentals(tickers: Dict[str, str]) -> Optional[pd.DataFrame]:
    """
    yfinance 대신 pykrx를 사용하여 한국거래소 공식 펀더멘털 지표 추출
    """
    today = datetime.today().strftime("%Y%m%d")
    results = []
    
    try:
        # KOSPI 전체 종목의 펀더멘털 데이터 일괄 조회
        krx_fundamentals = stock.get_market_fundamental(today, market="KOSPI")
        
        for name, ticker_code in tickers.items():
            if ticker_code in krx_fundamentals.index:
                data = krx_fundamentals.loc[ticker_code]
                
                per = data['PER']
                pbr = data['PBR']
                div_yield = data['DIV']
                
                # 딥밸류 필터링 (PER 0 초과 8 미만, PBR 1.0 미만)
                is_deep_value = "Yes" if (0 < per < 8) and (0 < pbr < 1.0) else "No"
                
                results.append({
                    '종목명': name,
                    'PER (배)': round(per, 2),
                    'PBR (배)': round(pbr, 2),
                    '배당수익률 (%)': round(div_yield, 2),
                    '극저평가 여부': is_deep_value
                })
            else:
                logging.warning(f"[{name}] 데이터를 찾을 수 없습니다.")
                
        return pd.DataFrame(results)

    except Exception as e:
        logging.error(f"KRX 데이터 추출 중 오류 발생: {e}")
        return None

if __name__ == "__main__":
    # pykrx는 '.KS' 접미사 없이 6자리 종목코드만 사용
    target_stocks = {
        "현대차": "053800",
        "KB금융": "105560",
        "삼성전자": "005930"
    }
    
    df = fetch_krx_fundamentals(target_stocks)
    if df is not None:
        print(df.to_string(index=False))

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.


--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\models.py", line 976, in json
    return complexjson.loads(self.text, **kwargs)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\json\decoder.py", line 345, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\json\decoder.py", line 363, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another exception occurred:

Trace

Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)


In [10]:
pip install --upgrade pykrx requests

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pykrx in c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages (1.2.8)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5




[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
# 필수 라이브러리 설치: pip install OpenDartReader yfinance pandas
import OpenDartReader
import yfinance as yf
import pandas as pd
import logging
from typing import Dict, Optional

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# TODO: 발급받은 DART API KEY를 여기에 입력하세요.
DART_API_KEY = "YOUR_DART_API_KEY_HERE"

def fetch_deep_value_via_dart(tickers: Dict[str, str], dart_key: str) -> Optional[pd.DataFrame]:
    """
    Open DART API (재무제표) + yfinance (주가) 결합 기반의 안정적인 퀀트 지표 산출
    """
    if dart_key == "YOUR_DART_API_KEY_HERE":
        logging.error("DART API 키가 설정되지 않았습니다.")
        return None

    dart = OpenDartReader(dart_key)
    results = []

    for name, ticker in tickers.items():
        try:
            # 1. DART: 기업의 발행주식총수 및 핵심 재무 데이터 (가장 최근 결산연도 기준)
            # 종목코드 6자리만 추출
            stock_code = ticker.replace(".KS", "") 
            
            # 단일 기업 주요 계정과목 (자본총계, 당기순이익) 추출
            # reprt_code="11011" (사업보고서)
            fin_data = dart.finstate_all(stock_code, "2023", reprt_code="11011") 
            
            if fin_data is None or fin_data.empty:
                logging.warning(f"[{name}] DART 재무 데이터를 불러올 수 없습니다.")
                continue

            # (연결재무제표 기준) 자본총계 및 당기순이익 파싱
            equity_row = fin_data.loc[(fin_data['sj_div'] == 'BS') & (fin_data['account_nm'].str.contains('자본총계'))]
            net_income_row = fin_data.loc[(fin_data['sj_div'] == 'IS') & (fin_data['account_nm'].str.contains('당기순이익'))]
            
            if equity_row.empty or net_income_row.empty:
               continue
               
            # string 형태의 금액을 int로 변환
            total_equity = int(equity_row.iloc[0]['thstrm_amount'])
            net_income = int(net_income_row.iloc[0]['thstrm_amount'])

            # 2. yfinance: 현재 주가 및 시가총액 데이터
            yf_stock = yf.Ticker(ticker)
            current_price = yf_stock.history(period="1d")['Close'].iloc[-1]
            market_cap = yf_stock.info.get('marketCap', 0)
            
            if market_cap == 0:
                continue

            # 3. 퀀트 지표 직접 산출 (PER, PBR, ROE)
            # PER = 시가총액 / 당기순이익
            per = market_cap / net_income if net_income > 0 else 0
            # PBR = 시가총액 / 자본총계
            pbr = market_cap / total_equity if total_equity > 0 else 0
            # ROE = 당기순이익 / 자본총계
            roe = (net_income / total_equity) * 100 if total_equity > 0 else 0
            
            # 딥밸류 조건: PER 8 미만, PBR 1.0 미만
            is_deep_value = "Yes" if (0 < per < 8) and (0 < pbr < 1.0) else "No"

            results.append({
                '종목명': name,
                '현재가': f"{int(current_price):,}원",
                'PER (배)': round(per, 2),
                'PBR (배)': round(pbr, 2),
                'ROE (%)': round(roe, 2),
                '극저평가 여부': is_deep_value
            })

        except Exception as e:
            logging.error(f"[{name}] 처리 중 에러 발생: {e}")
            continue

    if not results:
        return None

    df = pd.DataFrame(results)
    logging.info("Open DART 기반 가치평가 지표 추출 완료.")
    return df


if __name__ == "__main__":
    target_stocks = {
        "현대차": "053800.KS",
        "KB금융": "105560.KS",
        "삼성전자": "005930.KS"
    }
    
    metrics_df = fetch_deep_value_via_dart(target_stocks, DART_API_KEY)
    
    if metrics_df is not None:
        print("\n--- Deep Value 퀀트 지표 리포트 (DART API 기반) ---")
        print(metrics_df.to_string(index=False))

ModuleNotFoundError: No module named 'OpenDartReader'

In [21]:
import asyncio
import logging
import yfinance as yf
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional

# Logging 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

CACHE_DIR = Path("./quantum_stocks_cache")
CACHE_DIR.mkdir(exist_ok=True)

async def fetch_momentum_data(ticker: str, period: str = "1mo") -> Optional[pd.DataFrame]:
    """비동기 yfinance 호출을 통한 단기 모멘텀 및 거래량 데이터 수집"""
    cache_path = CACHE_DIR / f"{ticker}_{period}_momentum.parquet"
    
    try:
        # I/O Bound 작업 비동기 처리
        stock = yf.Ticker(ticker)
        df = await asyncio.to_thread(stock.history, period=period)
        
        if df.empty:
            logger.warning(f"No data retrieved for {ticker}")
            return None
            
        # 모멘텀 및 거래량 급등 지표 계산
        df['Returns'] = df['Close'].pct_change()
        df['Vol_MA5'] = df['Volume'].rolling(window=5).mean()
        
        # 거래량이 최근 5일 이동평균 대비 2배 이상 터진 유의미한 상승인지 검증
        df['Vol_Spike'] = df['Volume'] > (df['Vol_MA5'] * 2.0)
        
        df.to_parquet(cache_path)
        logger.info(f"Successfully processed {ticker}. Recent return: {df['Returns'].iloc[-1]:.2%}")
        return df
        
    except Exception as e:
        logger.error(f"Failed to fetch data for {ticker}: {e}")
        return None

async def analyze_quantum_stocks(tickers: List[str]) -> Dict[str, pd.DataFrame]:
    """다수 종목의 모멘텀 지표 동시 분석"""
    tasks = [fetch_momentum_data(ticker) for ticker in tickers]
    results = await asyncio.gather(*tasks)
    return {ticker: df for ticker, df in zip(tickers, results) if df is not None}

# 실행 진입점
if __name__ == "__main__":
    target_tickers = ["IONQ", "RGTI"]
    # asyncio.run(analyze_quantum_stocks(target_tickers))

In [22]:
# 기존 코드 하단의 실행부 전체를 아래 코드로 교체합니다.

target_tickers = ["IONQ", "RGTI"]

try:
    # Jupyter/IPython 환경에서는 이벤트 루프가 이미 동작 중이므로 직접 await 호출
    results = await analyze_quantum_stocks(target_tickers)
    
    for ticker, df in results.items():
        print(f"\n[{ticker} Recent Analysis]")
        print(df[['Close', 'Returns', 'Vol_Spike']].tail(3))
        
except KeyboardInterrupt:
    logger.info("Execution cancelled by user.")
except Exception as e:
    logger.error(f"Critical execution failure: {e}")

2026-05-26 10:32:22,939 - ERROR - Failed to fetch data for IONQ: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
2026-05-26 10:32:22,948 - ERROR - Failed to fetch data for RGTI: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to ins

In [27]:
import asyncio
import yfinance as yf
import pandas as pd
import logging
from typing import Dict
from pathlib import Path

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

async def fetch_ticker_data(ticker: str, period: str = "3mo") -> pd.DataFrame:
    """비동기 방식으로 yfinance 데이터를 추출"""
    loop = asyncio.get_event_loop()
    try:
        stock = yf.Ticker(ticker)
        # yfinance는 동기 라이브러리이므로 run_in_executor 활용
        df = await loop.run_in_executor(None, stock.history, period)
        if df.empty:
            raise ValueError(f"{ticker} 데이터 응답 없음")
        return df[['Close']]
    except Exception as e:
        logging.error(f"[{ticker}] 데이터 수집 실패: {e}")
        return pd.DataFrame()

async def analyze_capital_rotation(tickers: Dict[str, str], benchmark: str = "^KS11") -> None:
    """KOSPI 벤치마크 대비 섹터별 상대강도(Relative Strength) 산출"""
    try:
        logging.info("벤치마크(KOSPI) 및 타겟 종목 데이터 수집 시작...")
        
        # 비동기 병렬 데이터 수집
        tasks = [fetch_ticker_data(benchmark)] + [fetch_ticker_data(t) for t in tickers.values()]
        results = await asyncio.gather(*tasks)
        
        bench_df = results[0]
        if bench_df.empty:
            raise RuntimeError("벤치마크 데이터 수집 실패로 분석을 중단합니다.")
            
        bench_return = (bench_df['Close'].iloc[-1] / bench_df['Close'].iloc[0]) - 1

        summary = []
        for i, (name, ticker) in enumerate(tickers.items()):
            df = results[i+1]
            if not df.empty:
                asset_return = (df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1
                # RS > 0 이면 시장 수익률 상회 (자본 유입)
                relative_strength = asset_return - bench_return 
                summary.append({
                    "종목명": name,
                    "3개월 수익률": f"{asset_return:.2%}",
                    "벤치마크 대비 RS": f"{relative_strength:+.2%}",
                    "자본 흐름 상태": "자본 유입 (주도주)" if relative_strength > 0 else "자본 유출 (소외주)"
                })

        report_df = pd.DataFrame(summary)
        logging.info("\n--- AI 자본 이동(Capital Rotation) 분석 결과 ---")
        print(report_df.to_string(index=False))

    except Exception as e:
        logging.error(f"분석 중 치명적 오류: {e}")

if __name__ == "__main__":
    target_portfolio = {
        "Phase 1 (SK하이닉스)": "000660.KS",
        "Phase 2 (HD현대일렉트릭)": "267260.KS",
        "Phase 2 (두산에너빌리티)": "034020.KS",
        "Phase 3 (현대차)": "053800.KS"
    }
    
    # 비동기 이벤트 루프 실행
    asyncio.run(analyze_capital_rotation(target_portfolio))

RuntimeError: asyncio.run() cannot be called from a running event loop

In [28]:
if __name__ == "__main__":
    target_portfolio = {
        "Phase 1 (SK하이닉스)": "000660.KS",
        "Phase 2 (HD현대일렉트릭)": "267260.KS",
        "Phase 2 (두산에너빌리티)": "034020.KS",
        "Phase 3 (현대차)": "053800.KS"
    }
    
    # Jupyter/IPython 환경에서는 기존 이벤트 루프를 그대로 활용 (asyncio.run 제거)
    await analyze_capital_rotation(target_portfolio)

2026-05-26 10:42:37,977 - INFO - 벤치마크(KOSPI) 및 타겟 종목 데이터 수집 시작...
2026-05-26 10:42:38,671 - INFO - 
--- AI 자본 이동(Capital Rotation) 분석 결과 ---


               종목명 3개월 수익률 벤치마크 대비 RS    자본 흐름 상태
  Phase 1 (SK하이닉스)  88.63%    +60.26% 자본 유입 (주도주)
Phase 2 (HD현대일렉트릭)   4.78%    -23.58% 자본 유출 (소외주)
 Phase 2 (두산에너빌리티)   8.00%    -20.37% 자본 유출 (소외주)
     Phase 3 (현대차)  -0.46%    -28.83% 자본 유출 (소외주)


In [29]:
pip install mlxtend networkx

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 17.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 51.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   ---------------------------------------- 0/2 [networkx]
   --


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import pandas as pd
data1 = pd.read_csv(r"C:\Users\itwill\Desktop\python\한글 텍스트 분석\구매내역.csv",encoding='cp949')
data1

,번호,고객명,구매항목
0,1,홍길동,새우깡
1,2,홍길동,맛동산
2,3,홍길동,맥주
3,4,일지매,짱구
4,5,일지매,감자깡
5,6,강감찬,감자깡
6,7,강감찬,새우깡
7,8,전우치,자갈치
8,9,전우치,맛동산
9,10,홍길동,짱구


In [31]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

tx = data1.groupby('고객명')['구매항목'].apply(list).tolist()

tx

[['감자깡', '새우깡', '포카칩', '맥주', '크라운산도'],
 ['자갈치', '짱구', '맛동산'],
 ['빠다코코낫', '맛동산'],
 ['짱구', '감자깡'],
 ['자갈치', '맛동산', '초코칩쿠키'],
 ['새우깡', '맛동산', '맥주', '짱구', '맛동산']]

In [32]:
te = TransactionEncoder()
te_ary = te.fit(tx).transform(tx)
df = pd.DataFrame(te_ary, columns=te.columns_)

df

,감자깡,맛동산,맥주,빠다코코낫,새우깡,자갈치,짱구,초코칩쿠키,크라운산도,포카칩
0,True,False,True,False,True,False,False,False,True,True
1,False,True,False,False,False,True,True,False,False,False
2,False,True,False,True,False,False,False,False,False,False
3,True,False,False,False,False,False,True,False,False,False
4,False,True,False,False,False,True,False,True,False,False
5,False,True,True,False,True,False,True,False,False,False


In [33]:
# apriori 알고리즘을 사용하여 빈발 항목 집합을 찾음
frequent_itemsets = apriori(df, min_support=0.05, use_colnames=True)

# 연관 규칙 생성
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# 결과 출력
display(rules)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({크라운산도}),frozenset({감자깡}),0.166667,0.333333,0.166667,1.0,3.0,1.0,0.111111,inf,0.8,0.50,1.0,0.750
1,frozenset({포카칩}),frozenset({감자깡}),0.166667,0.333333,0.166667,1.0,3.0,1.0,0.111111,inf,0.8,0.50,1.0,0.750
2,frozenset({빠다코코낫}),frozenset({맛동산}),0.166667,0.666667,0.166667,1.0,1.5,1.0,0.055556,inf,0.4,0.25,1.0,0.625
3,frozenset({자갈치}),frozenset({맛동산}),0.333333,0.666667,0.333333,1.0,1.5,1.0,0.111111,inf,0.5,0.50,1.0,0.750
4,frozenset({초코칩쿠키}),frozenset({맛동산}),0.166667,0.666667,0.166667,1.0,1.5,1.0,0.055556,inf,0.4,0.25,1.0,0.625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,"frozenset({감자깡, 맥주})","frozenset({크라운산도, 포카칩, 새우깡})",0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.00,1.0,1.000
150,"frozenset({포카칩, 맥주})","frozenset({감자깡, 크라운산도, 새우깡})",0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.00,1.0,1.000
151,"frozenset({감자깡, 포카칩})","frozenset({크라운산도, 새우깡, 맥주})",0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.00,1.0,1.000
152,frozenset({크라운산도}),"frozenset({감자깡, 포카칩, 새우깡, 맥주})",0.166667,0.166667,0.166667,1.0,6.0,1.0,0.138889,inf,1.0,1.00,1.0,1.000


In [35]:
 pip install pyvis


   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 756.0/756.0 kB 16.1 MB/s  0:00:00

   ---------------------------------------- 2/2 [pyvis]




[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# 한글 텍스트 연관규칙분석하기
import re
from pathlib import Path
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
REVIEW_PATH = r"C:\Users\itwill\Desktop\python\TXT 파일\A_reviews.txt"
STOPWORDS_PATH = r"C:\Users\itwill\Desktop\python\TXT 파일\stopwords.txt"
# ----------------------------
# 1) 데이터 로드
# ----------------------------
reviews = Path(REVIEW_PATH).read_text(encoding="utf-8").splitlines()
reviews = [r.strip() for r in reviews if r.strip()]
print(reviews)
stopwords = Path(STOPWORDS_PATH).read_text(encoding="utf-8").splitlines()
stopwords = set(w.strip() for w in stopwords if w.strip())
print(stopwords)


['이 크림 피부가 촉촉하고 보습력이 정말 좋아요', '수분감이 오래가고 피부가 진정되는 느낌이에요', '발림성도 괜찮고 피부가 하루 종일 촉촉해요', '약산성 제품이라 민감한 피부에도 부담 없이 사용할 수 있어요', '건조한 겨울에도 보습이 잘 유지되고 피부결이 매끄러워졌어요', '저자극 포뮬러라 붉은기가 줄어들고 편안하게 사용할 수 있어요', '촉촉한 수분크림 찾고 있다면 이 제품 강력 추천합니다', '밤에 바르고 자면 다음날 피부가 탱탱하고 촉촉해져 있어요', '끈적임이 거의 없고 흡수력이 좋아서 데일리로 쓰기 좋아요', '피부 진정 효과가 확실하고 속건조가 많이 개선됐어요', '수분 에센스랑 같이 사용하니 피부가 훨씬 부드러워졌어요', '순하고 촉촉해서 민감 피부도 안심하고 쓸 수 있어요', '보습 지속력이 좋아서 피부가 하루종일 편안해요', '피부 결이 정돈되고 매끄러워져서 메이크업도 잘 먹어요', '촉촉함이 오래가고 건조함이 확실히 줄었어요', '끈적이지 않고 산뜻하게 수분이 채워지는 느낌이에요', '흡수력이 뛰어나서 여러 번 덧발라도 부담이 없어요', '민감한 피부인데도 자극 없이 편안하게 사용 가능했어요', '촉촉하고 수분이 꽉 찬 느낌이라 만족도가 높아요', '보습감이 풍부하면서도 가볍게 발려서 너무 좋아요']
{'일리', '조금', '그런', '진짜', '높다', '정말', '바쁘다', '돼다', '덕임', '도움', '찾다', '구매', '있다', '거의', '다음', '없다', '좋다', '제품', '여러', '저렇게', '거', '하게', '효과', '이번', '때문', '재다', '하지만', '않다', '그리고', '사용', '가다', '저런', '있습니다', '있는', '먹다', '이렇게', '들다', '되다', '하다', '너무', '생각', '있어요', '해서', '은기', '자다', '사용감', '이런', '정도', '성도', '쓸다', '것', '좀', '그냥'}


In [42]:
# --------------------------------------------------------------------------
# 2) 토큰화 + 불용어 제거
# - 한글/영문/숫자만 남기고 분리
# - 길이 2 미만 토큰 제거(예: "좋", "요" 같은 잡토큰 방지)
# -------------------------------------------------------------------------
def tokenize(text: str):
 text = text.lower( ) # 모든 문자를 소문자로 변환
 text = re.sub(r"[^0-9a-z가-힣\s]", " ", text) # 특수문자 제거
 tokens = text.split()
 tokens = [
 t for t in tokens
 if len(t) >= 2 and t not in stopwords
 ]
 # 한 문장 안에서 같은 단어 중복은 연관규칙에 의미 없으니 set 처리
 return sorted(set(tokens))
transactions = [tokenize(r) for r in reviews]
transactions = [t for t in transactions if t] # 빈 거래 제거
print(f"리뷰 수: {len(reviews)} / 거래(토큰 존재) 수: {len(transactions)}")
print("샘플 거래 3개:", transactions[:3])

리뷰 수: 20 / 거래(토큰 존재) 수: 20
샘플 거래 3개: [['보습력이', '좋아요', '촉촉하고', '크림', '피부가'], ['느낌이에요', '수분감이', '오래가고', '진정되는', '피부가'], ['괜찮고', '발림성도', '종일', '촉촉해요', '피부가', '하루']]


In [47]:
# ---------------------------------------------
# 3) 원-핫 인코딩 후 Apriori(아프리오리)
# ----------------------------------------------
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)


In [46]:
MIN_SUPPORT = 0.05 # 예: 전체 거래의 5% 이상에서 등장
MIN_CONFIDENCE = 0.50
freq = apriori(df, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
freq["itemsets"] = freq["itemsets"].apply(lambda x: tuple(sorted(x)))
freq = freq.sort_values("support", ascending=False).reset_index(drop=True)
print("\n[자주 같이 등장한 단어 묶음(빈발 아이템셋)]")
print(freq.head(30))


[자주 같이 등장한 단어 묶음(빈발 아이템셋)]
    support          itemsets
0      0.30            (피부가,)
1      0.15            (좋아요,)
2      0.10           (흡수력이,)
3      0.10            (민감한,)
4      0.10            (좋아서,)
5      0.10            (사용할,)
6      0.10         (민감한, 없이)
7      0.10           (촉촉하고,)
8      0.10             (없이,)
9      0.10            (수분이,)
10     0.10           (편안하게,)
11     0.10          (느낌이에요,)
12     0.10           (오래가고,)
13     0.10             (피부,)
14     0.05          (개선됐어요,)
15     0.05            (건조한,)
16     0.05           (건조함이,)
17     0.05           (겨울에도,)
18     0.05             (결이,)
19     0.05            (괜찮고,)
20     0.05           (끈적이지,)
21     0.05           (끈적임이,)
22     0.05            (높아요,)
23     0.05           (느낌이라,)
24     0.05   (자면, 촉촉해져, 피부가)
25     0.05             (밤에,)
26     0.05  (좋아요, 촉촉하고, 피부가)
27     0.05            (가볍게,)
28     0.05           (덧발라도,)
29     0.05           (데일리로,)


In [50]:
# ----------------------------
# 4) 연관규칙 생성
# ----------------------------
rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
# 보기 좋게 정렬/컬럼 정리
rules["antecedents"] = rules["antecedents"].apply(lambda x: tuple(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: tuple(sorted(x)))
rules = rules.sort_values(["lift", "confidence", "support"],
ascending=False).reset_index(drop=True)
print("\n[연관규칙 Top]")
display(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(30))


[연관규칙 Top]


,antecedents,consequents,support,confidence,lift
0,"(자면, 피부가)","(촉촉해져,)",0.05,1.0,20.0
1,"(촉촉해져, 피부가)","(자면,)",0.05,1.0,20.0
2,"(자면,)","(촉촉해져, 피부가)",0.05,1.0,20.0
3,"(촉촉해져,)","(자면, 피부가)",0.05,1.0,20.0
4,"(자극,)","(가능했어요,)",0.05,1.0,20.0
5,"(가능했어요,)","(자극,)",0.05,1.0,20.0
6,"(피부인데도,)","(가능했어요,)",0.05,1.0,20.0
7,"(가능했어요,)","(피부인데도,)",0.05,1.0,20.0
8,"(가볍게,)","(발려서,)",0.05,1.0,20.0
9,"(발려서,)","(가볍게,)",0.05,1.0,20.0


In [51]:
# ----------------------------
# 5) 결과 저장(선택)
# ----------------------------
OUT_DIR = Path(r"c:\py_temp")
OUT_DIR.mkdir(parents=True, exist_ok=True)
freq.to_csv(OUT_DIR / "frequent_itemsets.csv", index=False, encoding="utf-8-sig")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].to_csv(
 OUT_DIR / "association_rules.csv", index=False, encoding="utf-8-sig"
)
print("\n저장 완료:", OUT_DIR / "frequent_itemsets.csv", OUT_DIR / "association_rules.csv")


저장 완료: c:\py_temp\frequent_itemsets.csv c:\py_temp\association_rules.csv


In [52]:
# ------------------------------------------------------------------
# 6) "함께 많이 언급되는 단어쌍"만 뽑고 싶으면(2-아이템셋)
# -----------------------------------------------------------------
pairs = freq[freq["itemsets"].apply(len) == 2].copy()
pairs[["w1", "w2"]] = pairs["itemsets"].apply(lambda t: pd.Series(t))
pairs = pairs[["w1", "w2", "support"]].sort_values("support", ascending=False)
pairs.to_csv(OUT_DIR / "top_pairs.csv", index=False, encoding="utf-8-sig")
print("단어쌍 저장:", OUT_DIR / "top_pairs.csv")

단어쌍 저장: c:\py_temp\top_pairs.csv


In [54]:
# 단어 통일하도록 개선된 버전 – 더 정확한 방식
# pip install konlpy jpype1 pandas mlxtend
import re
from pathlib import Path
import pandas as pd
from konlpy.tag import Okt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
REVIEW_PATH = r"C:\Users\itwill\Desktop\python\TXT 파일\A_reviews.txt"
STOPWORDS_PATH = r"C:\Users\itwill\Desktop\python\TXT 파일\stopwords.txt"
okt = Okt()
reviews = Path(REVIEW_PATH).read_text(encoding="utf-8").splitlines()
reviews = [r.strip() for r in reviews if r.strip()]
stopwords = set(
 w.strip() for w in Path(STOPWORDS_PATH).read_text(encoding="utf-8").splitlines()
 if w.strip()
)
print(stopwords)

{'일리', '조금', '그런', '진짜', '높다', '정말', '바쁘다', '돼다', '덕임', '도움', '찾다', '구매', '있다', '거의', '다음', '없다', '좋다', '제품', '여러', '저렇게', '거', '하게', '효과', '이번', '때문', '재다', '하지만', '않다', '그리고', '사용', '가다', '저런', '있습니다', '있는', '먹다', '이렇게', '들다', '되다', '하다', '너무', '생각', '있어요', '해서', '은기', '자다', '사용감', '이런', '정도', '성도', '쓸다', '것', '좀', '그냥'}


In [55]:
# 각 행별로 형태소 분석
def tokenize_okt(text: str):
 text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
 # norm/stem으로 정규화/어간추출(예: 좋아요 -> 좋다 형태로)
 tokens = okt.pos(text, norm=True, stem=True)
 # 명사/형용사/동사만 남기는 예시 (필요 시 조정)
 keep_tags = {"Noun", "Adjective", "Verb"}
 words = [w for w, tag in tokens if tag in keep_tags]
 words = [
 w.lower() for w in words
 if len(w) >= 2 and w not in stopwords
 ]
 return sorted(set(words))
transactions = [tokenize_okt(r) for r in reviews]
print(transactions)

[['보습', '촉촉하다', '크림', '피부'], ['느낌', '수분', '진정', '피부'], ['괜찮다', '발림', '종일', '촉촉하다', '피부', '하루'], ['민감하다', '부담', '산성', '피부'], ['건조하다', '겨울', '매끄럽다', '보습', '유지', '피부'], ['뮬러', '붉다', '자극', '줄어들다', '편안하다'], ['강력', '수분크림', '촉촉하다', '추천'], ['바르다', '촉촉하다', '탱탱하다', '피부'], ['쓰기', '적임', '흡수'], ['개선', '건조', '진정', '피부', '확실하다'], ['부드럽다', '수분', '에센스', '피부'], ['민감', '순하다', '안심', '촉촉하다', '피부'], ['보습', '종일', '지속', '편안하다', '피부', '하루'], ['매끄럽다', '메이크업', '정돈', '피부'], ['건조하다', '줄다', '촉촉하다', '확실하다'], ['느낌', '산뜻하다', '수분', '채우다'], ['뛰어나다', '발라', '부담', '흡수'], ['가능하다', '민감하다', '자극', '편안하다', '피부'], ['느낌', '만족도', '수분', '촉촉하다'], ['가볍다', '발리다', '보습', '풍부하다']]


In [56]:
# 연관 규칙 찾기
transactions = [t for t in transactions if t]
te = TransactionEncoder()
df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
freq = apriori(df, min_support=0.05, use_colnames=True, max_len=3)
rules = association_rules(freq, metric="confidence", min_threshold=0.50)
rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)
display(rules[["antecedents","consequents","support","confidence","lift"]].head(30))

,antecedents,consequents,support,confidence,lift
6,frozenset({발리다}),frozenset({가볍다}),0.05,1.0,20.0
7,frozenset({가볍다}),frozenset({발리다}),0.05,1.0,20.0
9,frozenset({풍부하다}),frozenset({가볍다}),0.05,1.0,20.0
10,frozenset({가볍다}),frozenset({풍부하다}),0.05,1.0,20.0
11,frozenset({강력}),frozenset({수분크림}),0.05,1.0,20.0
12,frozenset({수분크림}),frozenset({강력}),0.05,1.0,20.0
14,frozenset({강력}),frozenset({추천}),0.05,1.0,20.0
15,frozenset({추천}),frozenset({강력}),0.05,1.0,20.0
16,frozenset({건조}),frozenset({개선}),0.05,1.0,20.0
17,frozenset({개선}),frozenset({건조}),0.05,1.0,20.0


In [64]:
# pyvis 시각화하기
import math
import networkx as nx
from pyvis.network import Network

# rules: mlxtend.association_rules 로 만든 DataFrame을 그대로 사용
# rules 컬럼: antecedents, consequents, support, confidence, lift ...
def rules_to_pyvis(
    rules,
    out_html=r"c:\py_temp\arules_network.html",
    min_lift=1.0,
    min_conf=0.5,
    top_k=80, # 너무 많으면 복잡하니 상위 k개만
    height="750px",
    width="100%",
):
    # -----------------------------
    # 1) 필터 + 정렬 + Top-K
    # -----------------------------
    rr = rules.copy()
    rr = rr[(rr["lift"] >= min_lift) & (rr["confidence"] >= min_conf)]
    rr = rr.sort_values(["lift", "confidence", "support"], ascending=False).head(top_k)

    # ---------------------------------------------------------------------------
    # 2) Graph 구성 (antecedents -> consequents)
    # - antecedents가 여러 단어면 "A+B" 형태로 묶어서 노드화
    # ---------------------------------------------------------------------------
    G = nx.DiGraph()

    def as_label(x):
        # x는 frozenset/tuple 가능
        items = sorted(list(x))
        return " + ".join(items)

    for _, row in rr.iterrows():
        a = as_label(row["antecedents"])
        c = as_label(row["consequents"])
        support = float(row["support"])
        50
        conf = float(row["confidence"])
        lift = float(row["lift"])

        # 노드 추가(노드 크기는 나중에 degree로 반영)
        if a not in G:
            G.add_node(a)
        if c not in G:
            G.add_node(c)

        # 엣지 가중치: lift 기반 (선 굵기)
        # (너무 큰 lift는 과도하니 log로 완화)
        weight = math.log1p(lift)

        # tooltip에 지표를 넣기
        title = f"support={support:.3f}<br>confidence={conf:.3f}<br>lift={lift:.3f}"
        G.add_edge(
            a,
            c,
            weight=weight,
            title=title,
            support=support,
            confidence=conf,
            lift=lift
        )

In [69]:
# pyvis 시각화하기
import math
import networkx as nx
from pyvis.network import Network

# rules: mlxtend.association_rules 로 만든 DataFrame을 그대로 사용
# rules 컬럼: antecedents, consequents, support, confidence, lift ...
def rules_to_pyvis(
    rules,
    out_html=r"c:\py_temp\arules_network.html",
    min_lift=1.0,
    min_conf=0.5,
    top_k=80, # 너무 많으면 복잡하니 상위 k개만
    height="750px",
    width="100%",
):
    # -----------------------------
    # 1) 필터 + 정렬 + Top-K
    # -----------------------------
    rr = rules.copy()
    rr = rr[(rr["lift"] >= min_lift) & (rr["confidence"] >= min_conf)]
    rr = rr.sort_values(["lift", "confidence", "support"], ascending=False).head(top_k)

    # ---------------------------------------------------------------------------
    # 2) Graph 구성 (antecedents -> consequents)
    # - antecedents가 여러 단어면 "A+B" 형태로 묶어서 노드화
    # ---------------------------------------------------------------------------
    G = nx.DiGraph()

    def as_label(x):
        # x는 frozenset/tuple 가능
        items = sorted(list(x))
        return " + ".join(items)

    for _, row in rr.iterrows():
        a = as_label(row["antecedents"])
        c = as_label(row["consequents"])
        support = float(row["support"])
        conf = float(row["confidence"])
        lift = float(row["lift"])

        # 노드 추가(노드 크기는 나중에 degree로 반영)
        if a not in G:
            G.add_node(a)
        if c not in G:
            G.add_node(c)

        # 엣지 가중치: lift 기반 (선 굵기)
        # (너무 큰 lift는 과도하니 log로 완화)
        weight = math.log1p(lift)

        # tooltip에 지표를 넣기
        title = f"support={support:.3f}<br>confidence={conf:.3f}<br>lift={lift:.3f}"
        G.add_edge(
            a,
            c,
            weight=weight,
            title=title,
            support=support,
            confidence=conf,
            lift=lift
        )

    # -----------------------------------
    # 3) pyvis로 시각화
    # -----------------------------------
    net = Network(height=height, width=width, directed=True, bgcolor="#ffffff", font_color="#222")
    net.barnes_hut() # 자동 레이아웃

    # 노드 크기: degree 기반(연결 많은 노드 크게)
    deg = dict(G.degree())
    max_deg = max(deg.values()) if deg else 1

    for node in G.nodes():
        size = 10 + 25 * (deg.get(node, 0) / max_deg)
        net.add_node(node, label=node, title=f"degree={deg.get(node,0)}", size=size)

    # 엣지 두께: weight 기반
    edge_weights = nx.get_edge_attributes(G, "weight")
    max_weight = max(edge_weights.values()) if edge_weights else 1

    for u, v, data in G.edges(data=True):
        width = 1 + 6 * (data["weight"] / max(1e-9, max_weight))
        net.add_edge(u, v, title=data["title"], value=width) # value가 시각적 굵기에 반영됨

    # 클릭/드래그 옵션(원하면 조절 가능)
    net.set_options("""
    var options = {
    "nodes": {"shape": "dot"},
    "edges": {
    "arrows": {"to": {"enabled": true}},
    "smooth": {"type": "dynamic"}
    },
    "interaction": {"hover": true, "multiselect": true},
    "physics": {"stabilization": {"iterations": 200}}
    }
    """)

    net.write_html(out_html, open_browser=False)

    return out_html, rr


# 사용 예시
out_html, used_rules = rules_to_pyvis(
    rules,
    out_html=r"c:\py_temp\arules_network.html",
    min_lift=1.2,
    min_conf=0.5,
    top_k=60
)

print("HTML 저장:", out_html)
print("시각화에 사용된 규칙 수:", len(used_rules))

HTML 저장: c:\py_temp\arules_network.html
시각화에 사용된 규칙 수: 60


In [70]:
# 무방향성으로 중복되는 규칙을 1개로 통일하기
import pandas as pd
def merge_bidirectional_singletons(rules: pd.DataFrame, how="max"):
 """
 A->B, B->A 같은 상호 규칙을 (A,B) 한 건으로 합침.
 how: 'max' 또는 'mean' (confidence/lift 집계 방식)
 """
 r = rules.copy()
 # frozenset({a,b})로 무방향 키 생성 (A,B) == (B,A)
 r["a"] = r["antecedents"].apply(lambda s: list(s)[0] if len(s)==1 else None)
 r["b"] = r["consequents"].apply(lambda s: list(s)[0] if len(s)==1 else None)
 r = r.dropna(subset=["a","b"]).copy()
 r["pair_key"] = r.apply(lambda x: tuple(sorted([x["a"], x["b"]])), axis=1)
53
 agg = {
 "support": "max", # support는 보통 동일하거나 비슷해서 max 권장
 "confidence": how,
 "lift": how,
 }
 merged = (r.groupby("pair_key", as_index=False)
 .agg(agg))
 merged[["w1","w2"]] = pd.DataFrame(merged["pair_key"].tolist(), index=merged.index)
 merged = merged.drop(columns=["pair_key"])\
.sort_values(["lift","confidence","support"], ascending=False)
 return merged

IndentationError: unexpected indent (3647383350.py, line 15)

In [72]:
# 무방향성으로 중복되는 규칙을 1개로 통일하기
import pandas as pd

def merge_bidirectional_singletons(rules: pd.DataFrame, how="max"):
    """
    A->B, B->A 같은 상호 규칙을 (A,B) 한 건으로 합침.
    how: 'max' 또는 'mean' (confidence/lift 집계 방식)
    """
    r = rules.copy()

    # frozenset({a,b})로 무방향 키 생성 (A,B) == (B,A)
    r["a"] = r["antecedents"].apply(lambda s: list(s)[0] if len(s)==1 else None)
    r["b"] = r["consequents"].apply(lambda s: list(s)[0] if len(s)==1 else None)
    r = r.dropna(subset=["a","b"]).copy()
    r["pair_key"] = r.apply(lambda x: tuple(sorted([x["a"], x["b"]])), axis=1)

    agg = {
        "support": "max", # support는 보통 동일하거나 비슷해서 max 권장
        "confidence": how,
        "lift": how,
    }

    merged = (r.groupby("pair_key", as_index=False)
        .agg(agg))

    merged[["w1","w2"]] = pd.DataFrame(merged["pair_key"].tolist(), index=merged.index)

    merged = merged.drop(columns=["pair_key"])\
        .sort_values(["lift","confidence","support"], ascending=False)

    return merged

# 결과
merged_pairs = merge_bidirectional_singletons(rules, how="max")
display(merged_pairs.head(30))

,support,confidence,lift,w1,w2
4,0.05,1.0,20.0,가볍다,발리다
6,0.05,1.0,20.0,가볍다,풍부하다
7,0.05,1.0,20.0,강력,수분크림
9,0.05,1.0,20.0,강력,추천
10,0.05,1.0,20.0,개선,건조
27,0.05,1.0,20.0,겨울,유지
29,0.05,1.0,20.0,괜찮다,발림
39,0.05,1.0,20.0,뛰어나다,발라
49,0.05,1.0,20.0,메이크업,정돈
51,0.05,1.0,20.0,뮬러,붉다


In [74]:
# pyvis 시각화하기
import networkx as nx
from pyvis.network import Network
import math
import pandas as pd

def pyvis_nodes_show_number(df_pairs, out_html=r"c:\py_temp\arules_num_nodes.html", top_k=80):
    d = (df_pairs
        .sort_values(["lift","confidence","support"], ascending=False)
        .head(top_k)
        .copy())

    d[["w1","w2"]] = d.apply(lambda x: pd.Series(sorted([x["w1"], x["w2"]])), axis=1)
    d = d.drop_duplicates(subset=["w1","w2"], keep="first")

    G = nx.Graph()

    for _, row in d.iterrows():
        w1, w2 = str(row["w1"]), str(row["w2"])
        lift = float(row["lift"])
        title = f"{w1} \
— {w2}<br>support={row['support']:.3f}<br>\
confidence={row['confidence']:.3f}<br>lift={lift:.3f}"
        G.add_edge(w1, w2, title=title, weight=math.log1p(lift))

    deg = dict(G.degree())
    max_deg = max(deg.values()) if deg else 1

    net = Network(height="750px", width="100%", directed=False, \
bgcolor="#ffffff", font_color="#222")
    net.barnes_hut()

    net.set_options(r"""
    var options = {
    "nodes": {
    "shape": "dot",
    "font": {"size": 22, "face": "Malgun Gothic", "color": "#111"}
    },
    "edges": {"smooth": {"type": "dynamic"}},
    "physics": {"stabilization": {"iterations": 300}}
    }
    """)

    # 노드 안에는 숫자(label), 단어는 title에
    for n in G.nodes():
        number = deg.get(n, 0) # 표시할 수치(예: degree)
        size = 15 + 35 * (number / max_deg) # size도 degree 기반

        # net.add_node(
        # n,
        # label=str(number), # 원 안에 숫자
        # title=f"{n}<br>degree={number}", # 단어는 hover로
        # size=size
        # )

        net.add_node(
            n,
            label=f"{n}\n{number}", # 단어 + 줄바꿈 + 숫자
            title=f"{n}<br>degree={number}",
            size=size
        )

    w = nx.get_edge_attributes(G, "weight")
    max_w = max(w.values()) if w else 1

    for u, v, data in G.edges(data=True):
        width_val = 1 + 8 * (data["weight"] / max_w)
        net.add_edge(u, v, title=data["title"], value=width_val)

    net.write_html(out_html, open_browser=False)

    return out_html


out_html = pyvis_nodes_show_number(merged_pairs, \
out_html=r"c:\py_temp\arules_num_nodes.html", top_k=80)

print("저장된파일명:", out_html)

저장된파일명: c:\py_temp\arules_num_nodes.html


In [10]:
from crawl4ai import AsyncWebCrawler

target_url = input("크롤링할 URL을 입력하세요: ").strip()

if not target_url:
    raise ValueError("URL이 비어 있습니다. 크롤링할 URL을 입력해야 합니다.")

if not target_url.startswith(("http://", "https://")):
    target_url = "https://" + target_url

async def main():
    print(f"[확인] 크롤링 URL: {target_url}")

    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url=target_url)

        print("\n" + "=" * 80)
        print("[크롤링 결과 - Markdown]")
        print("=" * 80)

        if result.markdown:
            print(result.markdown)
        else:
            print("[경고] markdown 결과가 비어 있습니다.")
            print("[디버그] result 객체:")
            print(result)

await main()

[확인] 크롤링 URL: https://www.nbcnews.com/business


NotImplementedError: 

In [12]:
import asyncio
import sys
from crawl4ai import AsyncWebCrawler


TARGET_URL = "https://www.nbcnews.com/business"


async def main():
    print("=" * 80)
    print("[환경 확인]")
    print(f"Python: {sys.version}")
    print(f"URL: {TARGET_URL}")
    print("=" * 80)

    try:
        async with AsyncWebCrawler() as crawler:
            result = await crawler.arun(url=TARGET_URL)

            print("\n" + "=" * 80)
            print("[크롤링 결과 - Markdown]")
            print("=" * 80)

            markdown = getattr(result, "markdown", None)

            if markdown:
                print(markdown)
            else:
                print("[경고] markdown 결과가 비어 있습니다.")
                print("[디버그] result 객체:")
                print(result)

    except Exception as e:
        print("\n" + "=" * 80)
        print("[에러 발생]")
        print("=" * 80)
        print(f"에러 타입: {type(e).__name__}")
        print(f"에러 내용: {e}")
        raise


if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [1]:
import numpy as np
from paddleocr import PaddleOCR

img_path = r"C:\Users\itwill\Desktop\python\딥러닝\영수증.png"

ocr = PaddleOCR(
    lang="korean",
    use_textline_orientation=True
)

result = ocr.predict(
    img_path,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

print("OCR 실행 완료")
print("결과 타입:", type(result))
print("결과 개수:", len(result))

if len(result) > 0:
    print("첫 번째 결과 타입:", type(result[0]))

c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creat

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)
